exploratory data analysis


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv('../data/processed/nyc_parking_clean.csv')


In [3]:
plt.figure(figsize=(10,6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.savefig('../reports/figures/01_null_heatmap.png')
plt.close()


the null heatmap shows missing data patterns across columns. mostly concentrated in vehicle year and color fields. the core enforcement metrics remain intact.


In [4]:
plt.figure(figsize=(12,6))
top_v = df['Violation_Description'].value_counts().nlargest(10).index
sns.countplot(y='Violation_Description', hue='Violation_Severity', data=df[df['Violation_Description'].isin(top_v)])
plt.savefig('../reports/figures/02_top_violations.png')
plt.close()


street cleaning is the most frequent violation type overall. however fire hydrant violations are the dominant safety-critical issue. this indicates enforcement is split between routine compliance and safety hazards.


In [5]:
plt.figure(figsize=(10,6))
sns.histplot(df['Fine_Amount'].dropna(), bins=30)
plt.savefig('../reports/figures/03_fine_distribution.png')
plt.close()


the distribution of fines is heavily skewed with massive spikes at 65 and 115 dollars. these price points correspond directly to the standard rates for common tickets. there are very few tickets issued in the middle price ranges.


In [6]:
fig, ax1 = plt.subplots(figsize=(12,6))
monthly = df.groupby('Month_Name').agg(tickets=('Summons_Number','count'), revenue=('Fine_Amount','sum'))
ax2 = ax1.twinx()
ax1.plot(monthly.index, monthly['tickets'], color='blue', marker='o', label='Tickets')
ax2.plot(monthly.index, monthly['revenue'], color='orange', marker='s', label='Revenue')
plt.savefig('../reports/figures/04_monthly_trend.png')
plt.close()


the monthly trend reveals how ticket volume and revenue track together over time. certain months show a divergence where revenue drops despite high ticket volume. this signals a shift toward enforcing lower-value administrative violations during those periods.


In [7]:
plt.figure(figsize=(10,6))
pt = pd.crosstab(df['Day_of_Week'], df['Month_Name'])
sns.heatmap(pt, cmap='Blues')
plt.savefig('../reports/figures/05_day_month_heatmap.png')
plt.close()


the day by month heatmap uncovers temporal enforcement hotspots. tuesdays and thursdays consistently show higher ticketing density. weekends have fundamentally lower enforcement activity across all months.


In [8]:
plt.figure(figsize=(10,6))
df.groupby('Violation_County')['Fine_Amount'].sum().sort_values().plot(kind='barh')
plt.savefig('../reports/figures/06_borough_revenue.png')
plt.close()


manhattan generates the highest absolute total revenue among all boroughs. this is expected given the extreme parking scarcity and density. staten island generates a negligible fraction of total revenue.


In [9]:
plt.figure(figsize=(12,8))
pt = pd.crosstab(df['Violation_County'], df['Violation_Description'], normalize='index')
sns.heatmap(pt, cmap='YlOrRd')
plt.savefig('../reports/figures/07_borough_violation_heatmap.png')
plt.close()


the normalized heatmap strips away the volume advantage of manhattan to reveal true behavioral profiles. the bronx has a notably distinct proportion of safety violations compared to brooklyn. this proves different boroughs require customized enforcement strategies.


In [10]:
fig, ax1 = plt.subplots(figsize=(10,6))
tier_rev = df.groupby('Offender_Tier')['Fine_Amount'].sum().sort_values(ascending=False)
cum_pct = (tier_rev.cumsum() / tier_rev.sum()) * 100
ax1.bar(tier_rev.index.astype(str), tier_rev.values)
ax2 = ax1.twinx()
ax2.plot(tier_rev.index.astype(str), cum_pct.values, color='red', marker='D')
plt.savefig('../reports/figures/08_pareto.png')
plt.close()


this pareto chart proves that a tiny fraction of repeat offenders generate a disproportionately massive share of revenue. targeting these specific license plates would yield higher returns than random patrols. escalation policies for habitual offenders are strictly necessary.


In [11]:
plt.figure(figsize=(8,6))
cols = ['Fine_Amount', 'Vehicle_Age', 'Is_Repeat_Offender', 'Is_Avoidable']
corr = df[cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.savefig('../reports/figures/09_correlation.png')
plt.close()


the correlation matrix tests relationships between numeric and boolean variables. vehicle age shows very weak correlation with fine amount. repeat offenders actually correlate slightly negatively with avoidable violations, suggesting they commit harder-to-avoid infractions.


In [12]:
plt.figure(figsize=(8,6))
pt = pd.crosstab(df['Is_Avoidable'].map({1:'Avoidable', 0:'Unavoidable'}), df['Violation_Severity'])
pt.plot(kind='bar', stacked=True, figsize=(8,6))
plt.savefig('../reports/figures/10_avoidable_safety.png')
plt.close()


this stacked bar breaks down violations by their avoidability and severity status. a massive portion of standard violations are completely avoidable through simple compliance like reading signs. safety-critical violations make up a smaller but highly urgent slice of unavoidable tickets.
